In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import torch
import cv2 as cv

In [2]:
pip install -U ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
from ultralytics import YOLO

# 1. Load pretrained weights
model = YOLO('/kaggle/working/yolov5s.pt') 

# 2. Start fine-tuning (Confidence is handled automatically)
model.train(
    data='/kaggle/input/datasets/nikolasgegenava/sard-search-and-rescue/search-and-rescue/data.yaml',  # Path to your SARD YAML file
    epochs=30,            # 30-50 epochs is usually sufficient for fine-tuning
    imgsz=1280,            # Increase to 1024 or 1280 if drone objects are very small
    batch= 4,
    workers=4
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
PRO TIP 💡 Replace 'model=/kaggle/working/yolov5s.pt' with new 'model=/kaggle/working/yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
Ultralytics 8.4.142 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutm

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/30      3.94G      2.204      2.414      2.023          2       1280: 100% ━━━━━━━━━━━━ 1011/1011 5.0it/s 3:220.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 6.0it/s 23.7s0.2s
                   all       1144       1463      0.529      0.407      0.427      0.159

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      3.94G      2.159        2.3      1.967          0       1280: 100% ━━━━━━━━━━━━ 1011/1011 5.0it/s 3:220.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 143/143 6.0it/s 23.9s0.2s
                   all       1144       1463      0.452      0.331      0.325      0.123

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      3.94G      2.069      2.165      1.921          3       1280: 100% ━━━━━━━━━━━━ 1011/1011 5.0it/s 3:210.4s
                 Class  

KeyboardInterrupt: 

In [5]:
import cv2
import torch
from ultralytics import YOLO

# 1. Force GPU cleanup before starting
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Load lightweight YOLO model
model = YOLO('/kaggle/input/models/chaitanya1121/best-wts/pytorch/default/1/best.pt')  # Use 'n' (nano) to start small and prevent memory crashes

video_path = '/kaggle/input/datasets/chaitanya1121/videooo/20260905-1246-11.9689341.mp4'
output_path = 'output_detected.mp4'

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise ValueError("Error opening video file")

# Get original video properties
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0

# Initialize VideoWriter
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

frame_count = 0

print("Processing video frames...")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run detection (device='cpu' if GPU memory is low, or device=0 for GPU)
    # Filter for human/person class (COCO class 0 = person)
    results = model.predict(source=frame, classes=[0], conf=0.25, verbose=False)

    # Plot detection bounding boxes directly onto frame
    annotated_frame = results[0].plot()

    # Write processed frame to disk instead of using cv2.imshow()
    out.write(annotated_frame)
    
    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processed {frame_count} frames...")
    if frame_count >=900:
        break

# Cleanup resources
cap.release()
out.release()
print(f"Processing complete! Saved to {output_path}")

Processing video frames...
Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...
Processed 210 frames...
Processed 240 frames...
Processed 270 frames...
Processed 300 frames...
Processed 330 frames...
Processed 360 frames...
Processing complete! Saved to output_detected.mp4
